# 1.📦 Importación de Librerías 📥📚
---
1. Traigo librerías para empezar, datos y scraping para conectar.
2. Sesión de requests lista quedó, y un plan de reintentos se definió
3. Si falla el Envío por un error, tres veces insiste con gran valor.
4. Se monta el adaptador con atención, aplicando la regla a la conexión.
5. Luego de que revisa, un mensaje avisa que todo cargó con prisa.
***

In [ ]:
# 1.1 Importacion de librerias necesarias
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import json
import re
from urllib.parse import urljoin,urlparse
from datetime import datetime
import pandas as pd
import os
from typing import Dict,List,Optional,Tuple,Any
import hashlib

#1.2 para manejar errores en red 
from request.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#1.3 Configuracion de sesion con reintentos
session = requests.Session()
retry=Retry(total=3, backoff_factor=1, status_forcelist=[429,500,502,503,504])

#1.4 adaptador
adaptador=HTTPAdapter(max_retries=retry)
session.mount('https://', adaptador)

#1.5crear carpetas datos si no existe
os.makedirs('datos', exist_ok=True)

#1.6 aviso de librerias importadas correctamente 
print("Configuracion completada. librerias importadas")

# 2. 🛠️Funciones auxiliares (🌐HTML, ⏳espera, 💾caché)
- Toma la URL y el HTML a explorar, y un BeautifulSoup te da para analizar.
- Pausa la ejecución unos segundos al andar, para respetar límites y no sobrecargar.
- En memoria un diccionario va a crear, y repeticiones a la API logra evitar.

In [ ]:
#2.1 Obtiene el contenido html de una pagina web, una url y lo devuelve, un objeto beutifulsoup.
def obtener_html(url:str, timeout:int=10)->Optional[BeautifulSoup]:
    try:
        respuesta=sesion.get(url,timeout=timeout)
        respuesta.raise_for_status()
        soup=BeautifulSoup(respuesta.text,'html.parser')
        print(f"Contenido HTML obtenido correctamente de {url}")
        return soup
    except requests.exceptions.HTTPError as e:
        if e.response.status_code==404:
            print(f"⚠️ Página web no encontrada (404): {url}")
        else:
            print(f"Error HTTP Al obtener el contenido HTML {e.response.status_code} de: {url}")
    except requests.exceptions.Timeout:
        print(f"Tiempo de espera agotado al obtener el contenido HTML de: {url}")
    except requests.exceptions.RequestException as e:
        print(f" Error de red al obtener el contenido HTML de: {url}. Detalles: {e}")

#2.2 funcion para esperar un tiempo pausa para respetar limites de velocidad
def esperar(segundos: int =1):
    print(f"⏳ Esperando {segundos} segundos...")
    time.sleep(segundos)

#2.3 cache de autores en (memoria)
cache_autores={}
def cargar_cache_autores():
    global cache_autores
    archivo_cache='datos/cache_autores.csv'
    if os.path.exists(archivo_cache):
        df=pd.read_csv(archivo_cache)
        for _, row in df.iterrows():
            nombre=row['nombre']
            datos={
                'ano_nacimiento': row.get('ano_nacimiento'),
                'pais': row.get('pais'),
                'total_obras_conocidas': row.get('total_obras_conocidas',0),
                'api_id': row.get('api_id'),
                'api_source': row.get('api_source')
            }
        # Convertir NaN a None para campos vacios
        for key, value in datos.items():
            if pd.isna(value):
                datos[key] = None
        cache_autores[nombre]=datos
    print(f"✅ Cache de autores cargada desde {archivo_cache}. Total autores: ({len(cache_autores)} en registros)")

#2.4 funcion para guardar cache de autores en un archivo csv
def guardar_cache_autores():
    if not cache_autores:
        print("⚠️ No hay datos de autores para guardar en la cache.")
        return
    registros=[]
    for nombre, datos in cache_autores.items():
        if datos is None:
            continue
        registros.append({
            'nombre': nombre,
            'ano_nacimiento': datos.get('ano_nacimiento'),
            'pais': datos.get('pais'),
            'total_obras_conocidas': datos.get('total_obras_conocidas',0),
            'api_id': datos.get('api_id'),
            'api_source': datos.get('api_source')
        })
    df=pd.DataFrame(registros)
    df.to_csv('datos/cache_autores.csv', index=False, encoding="utf-8")
    print(f"✅ Cache de autores guardada en datos/cache_autores.csv. Total autores: ({len(registros)} en registros)")

#cargar cache al inicair (para no repetir llamadas a la api)
cargar_cache_autores()

# 3.💻 Creación de la  base de datos🗃️💾 (DDL "Definición de Datos Lenguaje")
- La función crear base datos entrará en acción.
- Construye las tablas mientras hablas:
- Categorías, libros y autores crear sin vacilar.
- Con autor_libro logra conectar
- Sus claves primarias y foráneas al relacionar.
- Genera como vez sus índices con gran agilidad
- Para buscar todo con más velocidad encontrando su identidad.
- Con un commit guarda cada elemento,
- cierra la conexión y confirma el argumento del documento al momento.

In [ ]:
def crear_base_datos():
#3.1 abre la base de datos, preparan un cursor para ejecutar SQL y activan la integridad referencial entre tablas.
    conn=sqlite3.connect('libreria')
    cursor=conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON")

#3.2.1 tabla categorias con sqlite3
    cursor.execute('''
                CREATE TABLE IF NOT EXISTS categorias (
        id_categoria INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_categoria TEXT UNIQUE NOT NULL
        )
    ''')

#3.2.2 tabla libros
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS libros(
            id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo_libro TEXT NOT NULL,
            precio_libro REAL NOT NULL,
            calificacion_libro INTEGER,
            categoria_id INTEGER,
            url_libro TEXT UNIQUE,
            FOREING KEY (categoria_id) REFERENCES categorias(id_categoria)
                )
    ''')

#3.2.3 tabla autores
    cursor.execute('''
        CREATE  TABLE IF NOT EXISTS autores(
            id_autor INTEGER PRIMARY KEY AUTOINCREMENTE,
            nombre_autor TEXT NOT NULL,
            ano_nacimiento_autor INTEGER,
            pais_autor TEXT,
            api_external_id TEXT,
            total_obras_conocidas_autor INTEGER,
            api_source TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )    
    ''')

#3.2.4 tablas intermedias relacion muchos a muchos
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS libro_autor (
            id_libro INTEGER,
            id_autor INTEGER,
            PRIMARY KEY (id_libro, id_autor),
            FOREIGN KEY (id_libro) REFERENCES libros(id_libro),
            FOREIGN KEY (id_autor) REFERENCES autores(id_autor)
        )
    ''')

#3.3 indices para optimizar consultas
    cursor.execute('CREATE INDEX IF NOT EXISTS id_libros_categoria ON libros(categoria_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS id_autores_nombre ON autores(nombre_autor)')
    cursor.execute('CREATE INDEX IF NOT EXISTS id_libros_calificacion ON libros(calificacion_libro)')
    cursor.execute('CREATE INDEX IF NOT EXISTS id_autores_pais ON autores(pais_autor)')

#3.4 commit y cerrar conexion
    conn.commit()
    conn.close()
    print("✅ Base de datos y tablas creadas/verificadas.")

crear_base_datos()